# Active calibration (direction B) on UBP

For a new user, which images should they view during calibration so a decoder pretrained on other subjects adapts best,
counted in recorded presentations? Code: `activecal.py` (experiment) on top of `ubpkd.py` (UBP data, models, training),
both in this repo. Results go to `activecal.csv` on Drive and are committed to `results/` in the repo by the last cell.

**Run order:** 1–5 once per runtime, then 6 (diagnostics) → 7 (gate) → 8 (main, after setting the frozen adapter) → 9.
Every experiment cell is resumable: finished runs are skipped by tag. Use an L4 GPU with High-RAM.

In [ ]:
# 1. GPU, code (this repo, latest main), UBP code (pinned), packages. Private repo: add a GITHUB_TOKEN Colab secret.
import os
from google.colab import userdata
REPO = 'navyers21/eeg_active_calibration'
try:
    TOKEN = userdata.get('GITHUB_TOKEN'); URL = f'https://{TOKEN}@github.com/{REPO}'
except Exception:
    TOKEN, URL = None, f'https://github.com/{REPO}'
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2
!git clone -q {URL} /content/eac 2>/dev/null || git -C /content/eac pull -q
!git -C /content/eac log --oneline -1
!git clone -q https://github.com/HaitaoWuTJU/Uncertainty-aware-Blur-Prior /content/ubp 2>/dev/null || true
!git -C /content/ubp checkout -q 2e513ae && git -C /content/ubp log --oneline -1
!pip -q install open_clip_torch==2.29.0 einops omegaconf hf_transfer

In [ ]:
# 2. Configuration
SUBJECTS = list(range(1, 11))     # every subject: each non-target subject is a source for the LOSO decoder
SCREEN   = [2, 5, 9]              # screening subjects (targets), chosen in advance
TARGETS  = SCREEN + [8]
TUNE_SUB = 1                      # diagnostics and gate only, never a target
MONTAGES = ['occipital8', 'posterior17']   # Crown-8 dropped: f0 and ceiling both ~1%, no headroom
SEEDS    = [0, 1, 2]
SOURCE   = 'hf_whiten'            # rerun the winner with 'clean_mvnn' before any final channel claim
DRIVE_BASE = '/content/drive/MyDrive/eegdistillation_ubp'   # same folder as the UBP runs: reuses features + source models
HF_REPO  = 'Haitao999/things-eeg'
HF_LOCAL = '/content/data_hf'
RAW_DIR  = f'{DRIVE_BASE}/Raw_data'

In [ ]:
# 3. Drive: checkpoints, image features, results survive a runtime reset
from google.colab import drive
drive.mount('/content/drive')
RESULTS  = f'{DRIVE_BASE}/results.csv'          # ubpkd's registry (source-model training rows)
AL_CSV   = f'{DRIVE_BASE}/activecal.csv'        # this experiment
CKPT, FEATS = f'{DRIVE_BASE}/checkpoints', f'{DRIVE_BASE}/image_features'
for d in (DRIVE_BASE, CKPT, FEATS):
    os.makedirs(d, exist_ok=True)
print('results ->', AL_CSV)

In [ ]:
# 4. Data: UBP's processed EEG (per selected subject) + resized THINGS images from Hugging Face
import collections, glob, tarfile, zipfile, time
from huggingface_hub import list_repo_files, snapshot_download
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as e:
    print('No HF_TOKEN secret found, downloading unauthenticated:', e)

files = list_repo_files(HF_REPO, repo_type='dataset')
print(f'{len(files)} files in {HF_REPO}; top-level folders:')
for k, v in sorted(collections.Counter('/'.join(f.split('/')[:-1][:2]) for f in files).items()):
    print(f'  {v:6d}  {k}')

def prefix_of(key):
    hits = [f for f in files if key in f]
    return hits[0][:hits[0].index(key)] if hits else None

patterns = []
for key in ['Preprocessed_data_250Hz_whiten'] + (['Preprocessed_data_250Hz_no_whiten'] if SOURCE != 'hf_whiten' else []):
    p = prefix_of(key + '/')
    if p is None:
        print(f'!! {key} not found on the Hub' + (' (cell 5 builds it from raw data)' if 'no_whiten' in key else ''))
        continue
    patterns += [f'{p}{key}/sub-{s:02d}/*' for s in SUBJECTS]
p = prefix_of('Image_set_Resize')
assert p is not None, 'No Image_set_Resize on the Hub.'
patterns.append(f'{p}Image_set_Resize*')
assert any('whiten/' in x for x in patterns), 'Whitened EEG not found: check the listing above and edit the patterns.'

for attempt in range(8):
    try:
        snapshot_download(HF_REPO, repo_type='dataset', local_dir=HF_LOCAL,
                          allow_patterns=patterns, max_workers=4)
        break
    except Exception as e:
        if '429' not in str(e) and 'Too Many Requests' not in str(e):
            raise
        wait = 60 * (attempt + 1)
        print(f'Rate-limited by Hugging Face, retrying in {wait}s ...')
        time.sleep(wait)
else:
    raise RuntimeError('Still rate-limited after 8 attempts; wait a few minutes and re-run this cell.')

for a in glob.glob(f'{HF_LOCAL}/**/Image_set_Resize*.zip', recursive=True) + glob.glob(f'{HF_LOCAL}/**/Image_set_Resize*.tar*', recursive=True):
    print('extracting', a)
    (zipfile.ZipFile(a) if a.endswith('.zip') else tarfile.open(a)).extractall(os.path.dirname(a))

def find_dir(name):
    for root, dirs, _ in os.walk(HF_LOCAL):
        if name in dirs:
            return os.path.join(root, name)
WHITEN_DIR, IMAGE_ROOT = find_dir('Preprocessed_data_250Hz_whiten'), find_dir('Image_set_Resize')
NOWHITEN_DIR = find_dir('Preprocessed_data_250Hz_no_whiten') or '/content/data_nowhiten/Preprocessed_data_250Hz_no_whiten'
print('whitened EEG:', WHITEN_DIR, '\nimages:', IMAGE_ROOT, '\nun-whitened EEG:', NOWHITEN_DIR)
assert os.path.isdir(f'{IMAGE_ROOT}/train_images') and os.path.isdir(f'{IMAGE_ROOT}/test_images')
!du -sh {HF_LOCAL} && df -h /content | tail -1

In [ ]:
# 5. Initialise ubpkd + activecal, cache the two CLIP feature sets used here (once per Drive)
import sys, importlib, subprocess
sys.path.insert(0, '/content/eac')
import ubpkd as U, activecal as A
importlib.reload(U); importlib.reload(A)
U.setup(ubp_dir='/content/ubp', whiten_dir=WHITEN_DIR, nowhiten_dir=NOWHITEN_DIR, image_root=IMAGE_ROOT,
        cache_dir='/content/cache', feat_dir=FEATS, ckpt_dir=CKPT, results_csv=RESULTS, source=SOURCE,
        pool_subjects=SUBJECTS, num_workers=os.cpu_count(),
        ubp_commit=subprocess.run(['git', '-C', '/content/ubp', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())
A.ENV['results_csv'] = AL_CSV
for spec in ['none', A.TARGET_SPEC]:
    U.image_features(spec)
print('ready | budgets', A.BUDGETS, '| gate configs', [A.cfg_name(A.make_cfg(**c)) for c in A.GATE_CFGS])

In [ ]:
# 6. diag (tuning subject): split disjointness, EEG scale per subject, f0 on target vs sources, identity check
#    (adapter with 0 steps == f0), and full-pool ceilings: adapter, every layer from f0, target-only from scratch.
#    full_pool_scratch should be roughly in UBP's within-subject range; far lower = upstream problem, not calibration.
A.run_group('diag', [TUNE_SUB], MONTAGES)

In [ ]:
# 7. gate (tuning subject): random policy x adapter / acquisition settings, budgets up to 3,200, 2 seeds.
#    Pass = some setting >= f0 at the first budgets and several SEs above f0 by 400-800.
A.run_group('gate', [TUNE_SUB], MONTAGES, seeds=SEEDS[:2])
display(A.gate_table())

In [ ]:
# 8. main: freeze the best passing setting from cell 7, then every policy x seed for each target and montage.
#    Do not change the frozen adapter after looking at target results.
A.ENV['frozen'] = dict(adapter='all', anchor=1.0, lr=1e-4, acquire='image', rehearse=0.5)   # <- set from cell 7
print('frozen:', A.cfg_name(A.make_cfg(**A.ENV['frozen'])))
A.run_group('main', TARGETS, MONTAGES, seeds=SEEDS)

In [ ]:
# 9. Results (frozen adapter): references, AUC table, curves, plot; then copy the CSV into the repo and push it.
import matplotlib.pyplot as plt, shutil
table, curves, ref = A.summary()
print('references:'); print(ref.to_string())
print('\nPRECONDITION: random needs gain_over_f0 > 0, otherwise the policy comparison is not valid.\n')
print(table.to_string(index=False))
print('\nheld-out top-1 by budget:'); print(curves.to_string())
fig, axes = plt.subplots(1, len(MONTAGES), figsize=(6 * len(MONTAGES), 4), squeeze=False)
for ax, mon in zip(axes[0], MONTAGES):
    if mon in curves.index.get_level_values(0):
        for pol, row in curves.loc[mon].iterrows():
            ax.plot(row.index, row.values, marker='o', lw=2.5 if pol in ('random', 'typiclust') else 1.2, label=pol)
        ax.axhline(ref.xs(mon, level='montage')['f0'].mean(), color='k', ls='--', lw=1, label='f0')
    ax.set_xscale('log', base=2); ax.set_xlabel('presentations'); ax.set_ylabel('held-out top-1 (%)'); ax.set_title(mon)
axes[0][-1].legend(fontsize=8); plt.tight_layout(); plt.savefig(f'{DRIVE_BASE}/activecal_curves.png', dpi=150); plt.show()
if TOKEN:
    for f in [AL_CSV, f'{DRIVE_BASE}/activecal_curves.png']:
        shutil.copy(f, '/content/eac/results/')
    !cd /content/eac && git config user.email colab@run && git config user.name colab && git add results && git commit -qm "results: activecal" && git push -q && git log --oneline -1